<a href="https://colab.research.google.com/github/Kireeti2808/agri_doctor/blob/main/Agri_Doctor_followup_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agri-Doctor Pro — follow-up experiments (rice field adaptation)

**What this notebook does** (results go to `My Drive/agri_results_followup`; the earlier folder `agri_results_v2` is only read, never changed):

1. **Rice field split.** The 4,000+ Sethy et al. (2020) rice field photos (bacterial blight, blast, brown spot) are cleaned of near-copies of the laboratory images and split, with near-copies kept together, into a held-out **field test** set (300 per class) and a **field pool**.
2. **Field adaptation.** For each of the 3 seeds (42, 7, 2024) the rice model is retrained on the *same* laboratory training images as before **plus the field pool**. Before = the lab-only model of that seed (already on Drive); after = the adapted model. Both are tested on the same field test photos, and on the laboratory test split to check nothing was forgotten.
3. **Rice few-shot curve.** 0, 5, 10, 20, 50, 100 field photos per class from the pool, tested on the field test set.
4. **Gate analysis on the adapted models**, including per-quadrant labels so the quadrant-agreement rule can be re-scored (and a "healthy-neutral" variant tested).
5. **Gemini cross-check** with a model that the key can actually use (`gemini-3.8-flash` is tried first).
6. **Second country (optional, strongly recommended):** Dhan-Shomadhan field photos from Bangladesh — a source the adapted model has never seen, so it shows whether adaptation carries over to new fields. Mendeley blocks Colab, so download it once in your browser from https://data.mendeley.com/datasets/znsxdctwtt/1 ("Download All") and put the zip, unchanged, in **My Drive/agri_external**.
7. **Your own photos (optional, strongly recommended):** photos you took yourselves (or got from the agriculture department) on a phone in real fields. Put them in **My Drive/agri_own_photos/rice/<class>/** and **My Drive/agri_own_photos/maize/<class>/**, with folder names exactly as the class names (e.g. `Rice_Brown_Spot`, `Maize_Common_Rust`; full list printed by step 2). Even 20–30 per class is worth having: it is the only test under true farmer conditions.

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Key icon (left sidebar): `GEMINI_API_KEY` must have **Notebook access** switched on *for this notebook*.
- Then **Runtime → Run all**, and click **Allow** in the Google Drive popup (account karthikeynori9390).

**Time:** about 2.5–3 hours (training ~25 min per seed). Every step saves to Drive, so if Colab disconnects, just **Run all** again: finished steps are skipped.

In [3]:
# 0. Settings
SEEDS = [42, 7, 2024]                  # same seeds as the lab-only models on Drive
FIELD_TEST_FRACTION = 0.30             # share of field photo groups held out for testing
FIELD_TEST_PER_CLASS = 300             # cap on test photos per class (keeps runtime bounded)
FIELD_VAL_FRACTION = 0.15              # share of the field pool used for early stopping
DUP_THRESHOLD = 4                      # perceptual-hash bits; <= this counts as a copy
SPLIT_SEED = 0                         # fixed: the field test set is the same for every seed
EVAL_DEPLOYED_MODEL = True             # also test the currently deployed rice model (reference)
FEW_SHOTS = (0, 5, 10, 20, 50, 100)
RUN_GEMINI = True
GEMINI_PREFERRED = ['gemini-3.8-flash']
GEMINI_MAIZE_SAMPLE = 60               # of the 83 maize field photos
GEMINI_RICE_PER_CLASS = 20             # rice field test photos per class sent to Gemini
OTHER_SET_CAP = 600                    # cap for the Bangladesh set and for your own photos

In [4]:
# 1. GPU, Google Drive (required), repository, packages
!nvidia-smi -L || echo "NO GPU - set Runtime > Change runtime type > T4 GPU, then Run all again"
import os, sys, json, time, shutil, subprocess, random, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive')
if not BASE.exists():
    raise RuntimeError('STOP: Google Drive is not mounted. Runtime > Restart session, run this cell again and click Allow.')
PREV = BASE / 'agri_results_v2'          # earlier results (read only)
OUT = BASE / 'agri_results_followup'; OUT.mkdir(exist_ok=True)
missing = [str(PREV / f'rice_seed{s}' / n) for s in SEEDS for n in ('rice_model.tflite', 'split.json')
           if not (PREV / f'rice_seed{s}' / n).exists()]
if missing:
    raise RuntimeError('STOP: the lab-only rice models from the first run are not on Drive:\n' + '\n'.join(missing))
if not Path('/content/agri_doctor').exists():
    !git clone -q https://github.com/Kireeti2808/agri_doctor /content/agri_doctor
%cd /content/agri_doctor
!pip install -q kagglehub gdown scikit-learn scipy google-genai
for p in ('/content/agri_doctor', '/content/agri_doctor/training', '/content'):
    sys.path.insert(0, p)
import numpy as np
import tensorflow as tf
COMMIT = subprocess.check_output(['git', '-C', '/content/agri_doctor', 'rev-parse', 'HEAD']).decode().strip()
print('repo commit', COMMIT, '| TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('STOP: no GPU in this session, so training would take 15+ hours.\n'
                       'Fix: Runtime > Change runtime type > T4 GPU > Save, then Run all. If Colab says no GPU is available, '
                       'your free GPU quota is used up for today: try again in a few hours (finished steps are kept on Drive).')
def done(name): return (OUT / name).exists()
def save(name, obj): json.dump(obj, open(OUT / name, 'w'), indent=1, default=str)
save('environment.json', {'commit': COMMIT, 'tf': tf.__version__, 'seeds': SEEDS, 'started': time.ctime()})

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-e93d8207-c12c-5cf7-4b9b-ff09e5ad6c63)
Mounted at /content/drive
/content/agri_doctor
repo commit 291de37cec0e5ba3f22154951aff0e1ea1712c27 | TF 2.21.0 | GPU [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
%%writefile /content/exp_lib.py
"""Agri-Doctor Pro experiment helpers (data assembly, inference with the app's
own decision code, and statistics). Pure numpy/scipy apart from agri_core."""
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

import agri_core as ac

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


# ------------------------------------------------------------------ data
def norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")


def list_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)


def ahash(path, size=8):
    with Image.open(path) as im:
        px = np.asarray(im.convert("L").resize((size, size), Image.Resampling.BILINEAR), dtype=np.float32)
    return np.packbits(px.flatten() > px.mean())


def hash_many(paths):
    out, ok = [], []
    for p in paths:
        try:
            out.append(ahash(p)); ok.append(p)
        except Exception:
            pass
    return (np.stack(out) if out else np.zeros((0, 8), np.uint8)), ok


def min_hamming(train_h, h):
    if len(train_h) == 0:
        return 64
    return int(np.unpackbits(np.bitwise_xor(train_h, h), axis=1).sum(1).min())


def dedupe_against(items, train_paths, threshold=4):
    """Drop external images that are (near-)copies of any training image."""
    train_h, _ = hash_many(train_paths)
    kept, removed = [], []
    for path, label in items:
        try:
            h = ahash(path)
        except Exception:
            continue
        (removed if min_hamming(train_h, h) <= threshold else kept).append((path, label))
    return kept, removed


def collect_by_keywords(root, rules, require=None, exclude=()):
    """rules: list of (keywords tuple, label). A file is labelled by the first
    rule whose keywords all occur in its normalised parent-path; `require`
    keywords must occur somewhere in the path; `exclude` keywords must not."""
    items, unmatched = [], Counter()
    root = Path(root)
    for p in list_images(root):
        rel = p.relative_to(root)
        path_text = norm(str(rel.parent))
        full = norm(str(rel))
        if require and not all(r in full for r in require):
            continue
        if any(x in full for x in exclude):
            continue
        label = None
        for keys, lab in rules:
            if all(k in path_text for k in keys):
                label = lab
                break
        if label is None:
            unmatched[p.parent.name] += 1
            continue
        if label != "SKIP":
            items.append((p, label))
    return items, dict(unmatched)


# ------------------------------------------------------------- inference
def old_rule(quads, healthy_label):
    """First app version: most frequent non-healthy quadrant label above 50%."""
    bad = [r["label"] for r in quads.values()
           if r["conf"] > ac.DETECTION_THRESHOLD and ac.label_kind(r["label"]) != "healthy"]
    return Counter(bad).most_common(1)[0][0] if bad else healthy_label


def run_image(img, true_label, true_crop, model_crop, interp, classes, router=None, tag=""):
    """Everything the app computes for one photo, plus TTA predictions."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    probs = ac.predict_image(img, interp).astype(float)
    ac.check_probabilities(probs, model_crop)
    whole = ac.classify_image(img, interp, classes, model_crop)
    quads = ac.analyze_quadrants(img, interp, classes, model_crop)
    s = ac.summarize_diagnosis(whole, quads)
    tta = [ac.predict_image(t, interp).astype(float)
           for t in (ImageOps.mirror(img), ImageOps.flip(img))]
    healthy = next(c for c in classes if "Healthy" in c)
    row = {
        "tag": tag, "true": true_label, "true_crop": true_crop, "model_crop": model_crop,
        "probs": [round(float(x), 6) for x in probs],
        "pred": whole["label"], "conf": whole["conf"],
        "agreement": s["agreement"], "needs_review": s["needs_review"],
        "verdict_kind": s["verdict_kind"], "pred_old": old_rule(quads, healthy),
        "tta_probs": [[round(float(x), 6) for x in t] for t in tta],
    }
    if router is not None:
        r = ac.route_crop(ac.predict_image(img, router))
        row.update({"router_crop": r["crop"], "router_conf": r["confidence"],
                    "router_review": r["needs_review"]})
    return row


# ------------------------------------------------------------ statistics
def wilson(k, n, z=1.959964):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def pct(k, n):
    lo, hi = wilson(k, n)
    return {"k": int(k), "n": int(n), "pct": round(100 * k / n, 2) if n else None,
            "ci95": [round(100 * lo, 1), round(100 * hi, 1)] if n else None}


def mcnemar_exact(a_ok, b_ok):
    from scipy.stats import binomtest
    b = sum(1 for x, y in zip(a_ok, b_ok) if x and not y)
    c = sum(1 for x, y in zip(a_ok, b_ok) if y and not x)
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    return {"a_only_correct": b, "b_only_correct": c, "p_exact": float(p)}


def risk_coverage(score, correct):
    """Sort by descending confidence score; return coverage, selective risk."""
    score = np.asarray(score, float); correct = np.asarray(correct, bool)
    order = np.argsort(-score, kind="stable")
    c = correct[order]
    n = len(c)
    cov = np.arange(1, n + 1) / n
    risk = np.cumsum(~c) / np.arange(1, n + 1)
    return cov, risk


def aurc(score, correct):
    cov, risk = risk_coverage(score, correct)
    return float(risk.mean())


def bootstrap(fn, n, reps=1000, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(reps):
        idx = rng.integers(0, n, n)
        v = fn(idx)
        if v is not None and not (isinstance(v, float) and math.isnan(v)):
            vals.append(v)
    if not vals:
        return None
    return [round(float(np.percentile(vals, 2.5)), 4), round(float(np.percentile(vals, 97.5)), 4)]


def ece(probs, labels, bins=15):
    probs = np.asarray(probs, float); labels = np.asarray(labels)
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == labels)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def _softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(probs, labels):
    from scipy.optimize import minimize_scalar
    logits = np.log(np.clip(np.asarray(probs, float), 1e-12, 1))
    labels = np.asarray(labels)

    def nll(t):
        p = _softmax(logits / t)
        return -np.log(np.clip(p[np.arange(len(labels)), labels], 1e-12, 1)).mean()
    return float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)


def apply_temperature(probs, t):
    return _softmax(np.log(np.clip(np.asarray(probs, float), 1e-12, 1)) / t)


def crossfit_temperature_ece(probs, labels, seed=0):
    """2-fold cross-fitted temperature scaling on the evaluation set itself."""
    n = len(labels); idx = np.random.default_rng(seed).permutation(n)
    halves = [idx[: n // 2], idx[n // 2:]]
    out = np.zeros((n, np.asarray(probs).shape[1])); temps = []
    probs = np.asarray(probs, float); labels = np.asarray(labels)
    for a, b in [(0, 1), (1, 0)]:
        t = fit_temperature(probs[halves[a]], labels[halves[a]]); temps.append(t)
        out[halves[b]] = apply_temperature(probs[halves[b]], t)
    return ece(out, labels), temps


def scores_for(rows, classes):
    P = np.array([r["probs"] for r in rows])
    srt = np.sort(P, 1)
    T = np.array([r["tta_probs"] for r in rows])          # n x 2 x C
    pred = P.argmax(1)
    tta_agree = (T.argmax(2) == pred[:, None]).sum(1)      # 0..2
    tta_mean_conf = (T.mean(1) * 0.5 + P * 0.5)[np.arange(len(P)), pred]
    ent = -(P * np.log(np.clip(P, 1e-12, 1))).sum(1)
    return {
        "max_softmax": P.max(1),
        "margin": srt[:, -1] - srt[:, -2],
        "neg_entropy": -ent,
        "tta_mean_conf": tta_mean_conf,
        "tta_agree_plus_conf": tta_agree + P.max(1),
        "quadrant_agree_plus_conf": np.array([r["agreement"] for r in rows]) + P.max(1),
        "gate_pass_plus_conf": np.array([0 if r["needs_review"] else 1 for r in rows]) + P.max(1),
    }


def analyse(rows, classes, reps=1000):
    """Full analysis for one model on one evaluation set."""
    from sklearn.metrics import confusion_matrix, f1_score
    rows = [r for r in rows if r["true"] in classes]
    n = len(rows)
    if n == 0:
        return {"n": 0}
    y = np.array([classes.index(r["true"]) for r in rows])
    P = np.array([r["probs"] for r in rows])
    pred = P.argmax(1)
    correct = pred == y
    present = sorted(set(y.tolist()))
    res = {"n": n, "class_counts": {classes[i]: int((y == i).sum()) for i in present}}
    res["accuracy"] = pct(int(correct.sum()), n)
    res["macro_f1_present_classes"] = round(float(f1_score(y, pred, labels=present, average="macro", zero_division=0)), 4)
    res["per_class_recall"] = {classes[i]: pct(int(((pred == i) & (y == i)).sum()), int((y == i).sum())) for i in present}
    res["confusion"] = {"labels": classes, "matrix": confusion_matrix(y, pred, labels=list(range(len(classes)))).tolist()}
    res["predicted_kind"] = dict(Counter(ac.label_kind(classes[p]) for p in pred))
    old_ok = [r["pred_old"] == r["true"] for r in rows]
    res["old_quadrant_vote_accuracy"] = pct(sum(old_ok), n)
    res["new_vs_old_mcnemar"] = mcnemar_exact(correct.tolist(), old_ok)
    crop_rows = [r for r in rows if "router_crop" in r and "Weed" not in r["true"]]
    if crop_rows:
        m = len(crop_rows)
        res["router"] = {"crop_leaf_images": m, "called": dict(Counter(r["router_crop"] for r in crop_rows)),
                         "correct_crop": pct(sum(r["router_crop"] == r["true_crop"] for r in crop_rows), m),
                         "auto_accepted_wrong_crop": pct(sum((not r["router_review"]) and r["router_crop"] != r["true_crop"] for r in crop_rows), m)}
    # the app's review gate as an operating point
    flag = np.array([r["needs_review"] for r in rows])
    keep = ~flag
    err = ~correct
    gate = {"flagged": pct(int(flag.sum()), n),
            "errors_caught": pct(int((flag & err).sum()), int(err.sum())),
            "correct_flagged": pct(int((flag & correct).sum()), int(correct.sum())),
            "accuracy_unflagged": pct(int((keep & correct).sum()), int(keep.sum()))}
    # matched-coverage comparison with plain max-softmax thresholding
    cov = keep.mean()
    sc = scores_for(rows, classes)
    if 0 < keep.sum() < n:
        k = int(keep.sum())
        def matched(idx):
            c = correct[idx]; f = flag[idx]; m = sc["max_softmax"][idx]
            kk = int((~f).sum())
            if kk == 0 or kk == len(idx):
                return None
            top = np.argsort(-m, kind="stable")[:kk]
            return float(c[~f].mean() - c[top].mean())
        top = np.argsort(-sc["max_softmax"], kind="stable")[:k]
        gate["msp_accuracy_at_same_coverage"] = pct(int(correct[top].sum()), k)
        gate["gate_minus_msp_accuracy"] = round(float(correct[keep].mean() - correct[top].mean()), 4)
        gate["gate_minus_msp_ci95"] = bootstrap(matched, n, reps)
    gate["coverage"] = round(float(cov), 4)
    res["review_gate"] = gate
    # selective prediction baselines
    res["aurc_lower_is_better"] = {}
    for name, s in sc.items():
        res["aurc_lower_is_better"][name] = {
            "aurc": round(aurc(s, correct), 4),
            "ci95": bootstrap(lambda idx, s=s: aurc(s[idx], correct[idx]), n, reps)}
    res["risk_at_coverage"] = {}
    for name, s in sc.items():
        cov_, risk_ = risk_coverage(s, correct)
        res["risk_at_coverage"][name] = {str(c): round(float(risk_[max(0, int(math.ceil(c * n)) - 1)]), 4) for c in (0.5, 0.7, 0.8, 0.9, 1.0)}
    # calibration
    res["ece_raw"] = round(ece(P, y), 4)
    try:
        e, temps = crossfit_temperature_ece(P, y)
        res["ece_temperature_scaled_crossfit"] = round(e, 4)
        res["temperatures"] = [round(t, 3) for t in temps]
    except Exception as ex:
        res["ece_temperature_scaled_crossfit"] = str(ex)
    return res


# ---------------------------------------------------------------- gemini
def gemini_eval(rows_by_key, images_by_key, crop_of, api_key, model, out_path, per_crop=60,
                sleep_s=7.0, seed=0, log=print):
    """Blind Gemini assessment on a stratified sample, resumable via out_path."""
    import time
    done = {}
    if Path(out_path).exists():
        done = {d["key"]: d for d in json.load(open(out_path))}
    rng = random.Random(seed)
    by_crop = defaultdict(list)
    for key, row in rows_by_key.items():
        by_crop[crop_of(row)].append(key)
    sample = []
    for crop, keys in sorted(by_crop.items()):
        keys = sorted(keys); rng.shuffle(keys)
        sample += keys[:per_crop]
    for i, key in enumerate(sample):
        if key in done:
            continue
        row = rows_by_key[key]
        img = ImageOps.exif_transpose(Image.open(images_by_key[key])).convert("RGB")
        out = ac.assess_leaf_image(ac.image_for_assessment(img), api_key, model)
        rec = {"key": key, "ok": out.get("ok", False)}
        if out.get("ok"):
            a = out["assessment"]
            summary = {"primary": row["pred"], "verdict_kind": row["verdict_kind"]}
            cmp_ = ac.compare_assessments(row["model_crop"], summary, a)
            rec.update({"g_crop": a["crop"], "g_label": a["closest_label"], "g_certainty": a["certainty"],
                        "category": cmp_["category"], "supports_prescription": cmp_["supports_prescription"]})
        else:
            rec["error"] = str(out)[:300]
        done[key] = rec
        json.dump(list(done.values()), open(out_path, "w"), indent=1)
        if i % 10 == 0:
            log(f"gemini {i}/{len(sample)}")
        time.sleep(sleep_s)
    return list(done.values())


def gemini_analyse(records, rows_by_key):
    from scipy.stats import fisher_exact
    recs = [r for r in records if r.get("ok")]
    out = {"assessed": len(recs), "failed_calls": sum(1 for r in records if not r.get("ok"))}
    if not recs:
        return out
    ml_ok = [rows_by_key[r["key"]]["pred"] == rows_by_key[r["key"]]["true"] for r in recs]
    g_ok = [r["g_label"] == rows_by_key[r["key"]]["true"] for r in recs]
    crop_ok = [r["g_crop"] == rows_by_key[r["key"]]["true_crop"] for r in recs]
    out["ml_accuracy_on_sample"] = pct(sum(ml_ok), len(recs))
    out["gemini_label_accuracy"] = pct(sum(g_ok), len(recs))
    out["gemini_crop_accuracy"] = pct(sum(crop_ok), len(recs))
    cats = Counter(r["category"] for r in recs)
    out["categories"] = dict(cats)
    out["ml_accuracy_by_category"] = {c: pct(sum(o for o, r in zip(ml_ok, recs) if r["category"] == c), n)
                                      for c, n in cats.items()}
    agree = [r["category"] == ac.AGREEMENT for r in recs]
    a = sum(o and g for o, g in zip(ml_ok, agree)); b = sum((not o) and g for o, g in zip(ml_ok, agree))
    c = sum(o and not g for o, g in zip(ml_ok, agree)); d = sum((not o) and not g for o, g in zip(ml_ok, agree))
    out["agreement_vs_correct_2x2"] = [[a, b], [c, d]]
    out["fisher_p"] = float(fisher_exact([[a, b], [c, d]])[1])
    gate = [rows_by_key[r["key"]]["needs_review"] for r in recs]
    errs = [not o for o in ml_ok]
    vlm_flag = [r["category"] not in (ac.AGREEMENT,) for r in recs]
    out["errors_caught_by_gate_only"] = pct(sum(g and e for g, e in zip(gate, errs)), max(1, sum(errs)))
    out["errors_caught_by_vlm_only"] = pct(sum(v and e for v, e in zip(vlm_flag, errs)), max(1, sum(errs)))
    out["errors_caught_by_gate_or_vlm"] = pct(sum((g or v) and e for g, v, e in zip(gate, vlm_flag, errs)), max(1, sum(errs)))
    out["results_flagged_by_gate_or_vlm"] = pct(sum(g or v for g, v in zip(gate, vlm_flag)), len(recs))
    out["prescription_on_wrong_diagnosis"] = pct(sum(r["supports_prescription"] and e for r, e in zip(recs, errs)), len(recs))
    return out


# ------------------------------------------------- conformal methods
def _cp_upper(k, n, delta):
    """Clopper-Pearson one-sided upper bound on a binomial rate."""
    from scipy.stats import beta
    return 1.0 if k >= n else float(beta.ppf(1 - delta, k + 1, n - k))


def selective_risk_control(score, correct, alpha, delta=0.1, splits=20, seed=0):
    """Learn-then-test style: on a calibration half pick the lowest threshold whose
    Clopper-Pearson upper bound on selective error is <= alpha (fixed-sequence
    test from the most confident end); report risk and coverage on the other half."""
    score = np.asarray(score, float); correct = np.asarray(correct, bool)
    n = len(score); rng = np.random.default_rng(seed); out = []
    for _ in range(splits):
        idx = rng.permutation(n); cal, te = idx[: n // 2], idx[n // 2:]
        order = cal[np.argsort(-score[cal], kind="stable")]
        thr = None; errs = 0
        for j, i in enumerate(order, 1):
            errs += int(not correct[i])
            if _cp_upper(errs, j, delta) <= alpha:
                thr = score[i]
            elif j >= 10:
                # fixed-sequence testing stops at the first failure after a minimal sample
                if thr is not None:
                    break
        if thr is None:
            out.append({"coverage": 0.0, "risk": None}); continue
        keep = score[te] >= thr
        out.append({"coverage": float(keep.mean()),
                    "risk": float((~correct[te][keep]).mean()) if keep.any() else None})
    covs = [o["coverage"] for o in out]; risks = [o["risk"] for o in out if o["risk"] is not None]
    return {"alpha": alpha, "delta": delta, "splits": splits,
            "coverage_mean": round(float(np.mean(covs)), 4), "coverage_sd": round(float(np.std(covs)), 4),
            "test_risk_mean": round(float(np.mean(risks)), 4) if risks else None,
            "risk_violations": int(sum(r > alpha for r in risks)), "splits_with_any_coverage": len(risks)}


def conformal_sets(probs, labels, alpha=0.1, splits=20, seed=0):
    """Split-conformal (LAC) prediction sets: coverage guarantee 1-alpha."""
    P = np.asarray(probs, float); y = np.asarray(labels); n = len(y)
    rng = np.random.default_rng(seed); cov, size, single_correct = [], [], []
    for _ in range(splits):
        idx = rng.permutation(n); cal, te = idx[: n // 2], idx[n // 2:]
        s = 1 - P[cal, y[cal]]; m = len(cal)
        q = np.quantile(s, min(1.0, math.ceil((m + 1) * (1 - alpha)) / m), method="higher")
        sets = (1 - P[te]) <= q
        cov.append(float(sets[np.arange(len(te)), y[te]].mean()))
        size.append(float(sets.sum(1).mean()))
        single = sets.sum(1) == 1
        single_correct.append(float(single.mean()))
    return {"alpha": alpha, "target_coverage": 1 - alpha,
            "empirical_coverage_mean": round(float(np.mean(cov)), 4), "coverage_sd": round(float(np.std(cov)), 4),
            "mean_set_size": round(float(np.mean(size)), 3), "share_single_label_sets": round(float(np.mean(single_correct)), 4)}


# --------------------------------------------- learned combined gate
GATE_FEATURES = ["max_softmax", "margin", "neg_entropy", "tta_mean_conf", "tta_agree", "quadrant_agree", "gate_pass"]


def gate_features(rows, classes, gemini=None):
    P = np.array([r["probs"] for r in rows]); srt = np.sort(P, 1); pred = P.argmax(1)
    T = np.array([r["tta_probs"] for r in rows])
    feats = np.column_stack([
        P.max(1), srt[:, -1] - srt[:, -2], (P * np.log(np.clip(P, 1e-12, 1))).sum(1),
        (T.mean(1) * 0.5 + P * 0.5)[np.arange(len(P)), pred], (T.argmax(2) == pred[:, None]).sum(1),
        [r["agreement"] for r in rows], [0 if r["needs_review"] else 1 for r in rows]])
    names = list(GATE_FEATURES)
    if gemini is not None:
        cats = ["Agreement", "Partial agreement", "Disagreement", "Uncertain"]
        feats = np.column_stack([feats] + [[1.0 if gemini.get(r["tag"]) == c else 0.0 for r in rows] for c in cats])
        names += ["gemini_" + c.lower().replace(" ", "_") for c in cats]
    y = np.array([classes.index(r["true"]) for r in rows]); correct = pred == y
    return feats, correct, names


def learned_gate_cv(X, correct, folds=5, seed=0):
    """Cross-validated logistic-regression gate; returns out-of-fold scores."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    oof = np.zeros(len(correct))
    if correct.all() or (~correct).all() or min(correct.sum(), (~correct).sum()) < folds:
        return None
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, correct):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
        m.fit(X[tr], correct[tr]); oof[te] = m.predict_proba(X[te])[:, 1]
    return oof


def compare_gates(X, correct, names, reps=1000):
    oof = learned_gate_cv(X, correct)
    res = {"n": int(len(correct)), "errors": int((~correct).sum())}
    if oof is None:
        res["note"] = "too few errors or correct cases for 5-fold cross-validation"; return res
    msp = X[:, names.index("max_softmax")]
    res["aurc_max_softmax"] = round(aurc(msp, correct), 4)
    res["aurc_learned_gate"] = round(aurc(oof, correct), 4)
    res["aurc_difference_learned_minus_msp"] = round(res["aurc_learned_gate"] - res["aurc_max_softmax"], 4)
    res["difference_ci95"] = bootstrap(lambda i: aurc(oof[i], correct[i]) - aurc(msp[i], correct[i]), len(correct), reps)
    for a in (0.05, 0.10):
        res[f"risk_control_alpha{a}_msp"] = selective_risk_control(msp, correct, a)
        res[f"risk_control_alpha{a}_learned"] = selective_risk_control(oof, correct, a)
    return res


def leave_one_set_out(sets_X, sets_c):
    """Train the gate on all other field sets, test on the held-out one."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    out = {}
    for held in sets_X:
        tr = [k for k in sets_X if k != held]
        if not tr:
            continue
        Xtr = np.vstack([sets_X[k] for k in tr]); ctr = np.concatenate([sets_c[k] for k in tr])
        if ctr.all() or (~ctr).all():
            continue
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, ctr)
        s = m.predict_proba(sets_X[held])[:, 1]; c = sets_c[held]
        out[held] = {"trained_on": tr, "aurc_learned": round(aurc(s, c), 4),
                     "aurc_max_softmax": round(aurc(sets_X[held][:, 0], c), 4)}
    return out


# ------------------------------------------------ few-shot adaptation
def few_shot_curve(lab_X, lab_y, field_X, field_y, shots=(0, 5, 10, 20), repeats=5, seed=0):
    """Same frozen-backbone embeddings; linear head trained on lab images plus k
    field images per class; tested on the remaining field images."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    rng = np.random.default_rng(seed); classes = sorted(set(field_y.tolist())); out = {}
    for k in shots:
        accs = []
        for r in range(repeats if k else 1):
            pick = []
            for c in classes:
                idx = np.where(field_y == c)[0]
                if k:
                    if len(idx) <= k:
                        continue
                    pick += rng.choice(idx, k, replace=False).tolist()
            pick = np.array(pick, int); rest = np.setdiff1d(np.arange(len(field_y)), pick)
            Xtr = np.vstack([lab_X, field_X[pick]]) if len(pick) else lab_X
            ytr = np.concatenate([lab_y, field_y[pick]]) if len(pick) else lab_y
            m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0))
            m.fit(Xtr, ytr)
            accs.append(float((m.predict(field_X[rest]) == field_y[rest]).mean()))
        out[str(k)] = {"accuracy_mean": round(float(np.mean(accs)), 4),
                       "accuracy_sd": round(float(np.std(accs)), 4), "repeats": len(accs)}
    return out

Writing /content/exp_lib.py


In [6]:
%%writefile /content/followup_lib.py
"""Agri-Doctor Pro follow-up experiments: rice field adaptation, rice few-shot,
quadrant-agreement mechanism check and the Gemini cross-check.
Uses the app's own decision code (agri_core), the evaluation helpers (exp_lib)
and the rice training script's leakage-safe split (training/train_rice.py)."""
import json
import random
import re
import shutil
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

import agri_core as ac
import exp_lib as ex
import train_rice as tr

FIELD_CLASSES = ["Rice_Bacterial_Leaf_Blight", "Rice_Brown_Spot", "Rice_Leaf_Blast"]
SETHY_MAP = [(("tungro",), None), (("bacterial",), "Rice_Bacterial_Leaf_Blight"),
             (("blight",), "Rice_Bacterial_Leaf_Blight"), (("blast",), "Rice_Leaf_Blast"),
             (("brown",), "Rice_Brown_Spot")]


# ---------------------------------------------------------------- data
def stage_sethy(src_root, dest):
    """Copy the Sethy et al. (2020) field photos into dest/<RICE class>/; tungro is left out
    (the rice model has no tungro class)."""
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    counts, unmatched = Counter(), Counter()
    for f in sorted(Path(src_root).rglob("*")):
        if not f.is_file() or f.suffix.lower() not in ex.IMG_EXT:
            continue
        rel = ex.norm(f.parent.relative_to(src_root))
        target = "UNMATCHED"
        for keys, name in SETHY_MAP:
            if all(k in rel for k in keys):
                target = name
                break
        if target is None:
            continue
        if target == "UNMATCHED":
            unmatched[f.parent.name] += 1
            continue
        (dest / target).mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest / target / f"{ex.norm(f.parent.name)}__{f.name}")
        counts[target] += 1
    return dict(counts), dict(unmatched)


def field_split(items, lab_paths, threshold=4, test_fraction=0.3, per_class=300, seed=0):
    """1) drop field photos that are near-copies of any laboratory image;
    2) group near-copies inside the field set;
    3) per-class group split into a held-out field TEST part and a field POOL
       (adaptation training and few-shot shots come only from the pool);
    4) cap the test part at `per_class` photos per class.
    No near-duplicate group straddles test and pool."""
    kept, removed = ex.dedupe_against(items, lab_paths, threshold)
    kept = sorted(kept, key=lambda x: str(x[0]))
    groups = tr.group_near_duplicates(kept, threshold)
    sp = tr.split_groups(kept, groups, 0.0, test_fraction, seed)
    test_idx, pool_idx = sp["test"], sp["train"]
    rng = random.Random(seed)
    by = defaultdict(list)
    for i in test_idx:
        by[kept[i][1]].append(i)
    test_kept, test_unused = [], 0
    for lab in sorted(by):
        idx = sorted(by[lab])
        rng.shuffle(idx)
        test_kept += idx[:per_class]
        test_unused += max(0, len(idx) - per_class)
    pos = {i: k for k, i in enumerate(pool_idx)}
    pool_groups = [[pos[i] for i in g] for g in groups if g[0] in pos]
    assert all(all(i in pos for i in g) for g in groups if g[0] in pos), "a group straddles test and pool"
    out = {
        "test": [[str(kept[i][0]), kept[i][1]] for i in sorted(test_kept)],
        "pool": [[str(kept[i][0]), kept[i][1]] for i in pool_idx],
        "pool_groups": pool_groups,
    }
    out["report"] = {
        "field_photos_found": len(items), "removed_as_copies_of_lab_images": len(removed),
        "after_dedupe": len(kept), "near_duplicate_groups_in_field_set": sum(1 for g in groups if len(g) > 1),
        "test_fraction": test_fraction, "test_cap_per_class": per_class,
        "test_used": len(out["test"]), "test_left_unused": test_unused, "pool": len(out["pool"]),
        "test_class_counts": dict(Counter(l for _, l in out["test"])),
        "pool_class_counts": dict(Counter(l for _, l in out["pool"])),
        "seed": seed,
    }
    return out


DHAN_RULES = [(("tungro",), "SKIP"), (("turngo",), "SKIP"), (("brown",), "Rice_Brown_Spot"),
              (("scal",), "Rice_Leaf_scald"), (("blast",), "Rice_Leaf_Blast"),
              (("sheath",), "Rice_Sheath_Blight"), (("steath",), "Rice_Sheath_Blight")]


def find_dhan(ext_dir, work_dir):
    """Dhan-Shomadhan (Bangladesh, 2023): Mendeley blocks Colab, so it is used only if the
    team has downloaded it by hand into My Drive/agri_external. Returns field-background photos."""
    import zipfile
    work_dir = Path(work_dir)
    srcs = [p for p in Path(ext_dir).iterdir() if "dhan" in p.name.lower() or "znsxdctwtt" in p.name.lower()] if Path(ext_dir).exists() else []
    if not srcs:
        return [], "not found: download https://data.mendeley.com/datasets/znsxdctwtt/1 in a browser and put the zip in My Drive/agri_external"
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True)
    for s in srcs:
        if s.is_dir():
            shutil.copytree(s, work_dir / s.name)
        elif s.suffix.lower() == ".zip":
            with zipfile.ZipFile(s) as z:
                z.extractall(work_dir / s.stem)
    for _ in range(3):                       # nested zips
        for z in list(work_dir.rglob("*.zip")):
            with zipfile.ZipFile(z) as zz:
                zz.extractall(z.with_suffix(""))
            z.unlink()
    items, _ = ex.collect_by_keywords(work_dir, DHAN_RULES, require=("field",))
    return items, f"{len(items)} field-background photos"


def own_photos(root, classes):
    """Photos the team took itself: root/<exact class name>/*.jpg. Returns (items, unknown folders)."""
    root = Path(root)
    if not root.exists():
        return [], []
    items, unknown = [], []
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        if d.name in classes:
            items += [(p, d.name) for p in ex.list_images(d)]
        else:
            unknown.append(d.name)
    return items, unknown


def _shrink_one(job):
    src, dst, max_side = job
    try:
        with Image.open(src) as im:
            if max(im.size) <= max_side and im.format == "JPEG":
                return src                                  # already small: train on the original file
            im.draft("RGB", (max_side, max_side))           # fast JPEG decode at reduced size
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
            im.save(dst, "JPEG", quality=95)
        return dst
    except Exception:
        return src


def shrink_for_training(paths, cache_dir, max_side=448, workers=None):
    """Large field photos are decoded and resized on every epoch, which makes training very slow.
    Make a copy at most max_side px (training still resizes to 224 in its own pipeline);
    small JPEGs are used unchanged. Returns {original path: path to train on}."""
    import hashlib, os
    from concurrent.futures import ProcessPoolExecutor
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(cache_dir / (hashlib.sha1(str(p).encode()).hexdigest() + ".jpg")), max_side) for p in paths]
    todo = [j for j in jobs if not Path(j[1]).exists()]
    out = {j[0]: j[1] for j in jobs if Path(j[1]).exists()}
    with ProcessPoolExecutor(workers or os.cpu_count()) as pool:
        for j, r in zip(todo, pool.map(_shrink_one, todo, chunksize=32)):
            out[j[0]] = r
    return out


def pool_train_val(pool, pool_groups, val_fraction, seed):
    items = [(Path(p), l) for p, l in pool]
    sp = tr.split_groups(items, pool_groups, val_fraction, 0.0, seed)
    return sp["train"], sp["val"]


# ------------------------------------------------------------ inference
def run_image_q(img, true_label, interp, classes, router=None, tag="", crop="Rice"):
    """The app's computation (exp_lib.run_image) plus per-quadrant labels and
    probabilities, so other quadrant rules can be scored from the records."""
    row = ex.run_image(img, true_label, crop, crop, interp, classes, router=router, tag=tag)
    im = ImageOps.exif_transpose(img).convert("RGB")
    qp = [ac.predict_image(q, interp).astype(float) for q in ac.split_quadrants(im).values()]
    row["quad_labels"] = [classes[int(np.argmax(p))] for p in qp]
    row["quad_probs"] = [[round(float(x), 6) for x in p] for p in qp]
    return row


def eval_model(pairs, interp, classes, router, tag_prefix, log=print, crop="Rice"):
    rows, t0 = [], time.time()
    for i, (p, lab) in enumerate(pairs):
        try:
            with Image.open(p) as im:
                r = run_image_q(im, lab, interp, classes, router=router, tag=f"{tag_prefix}:{i}", crop=crop)
            r["file"] = str(p)
            rows.append(r)
        except Exception as e:
            log("skip", p, e)
        if i % 150 == 0:
            log(tag_prefix, i, "/", len(pairs), f"{time.time() - t0:.0f}s")
    return rows


# ------------------------------------------------------ quadrant scores
def quadrant_scores(rows, classes):
    """Variants of the quadrant evidence, each combined with max-softmax like the paper's score.
    healthy_neutral: a quadrant contradicts the verdict only if it names a DIFFERENT
    disease or weed; quadrants read as healthy are neutral (sparse lesions)."""
    P = np.array([r["probs"] for r in rows])
    pred = P.argmax(1)
    msp = P.max(1)
    Q = np.array([r["quad_probs"] for r in rows])                 # n x 4 x C
    qlab = [r["quad_labels"] for r in rows]
    contra = np.array([sum(1 for l in ql if l != classes[k] and ac.label_kind(l) != "healthy")
                       for ql, k in zip(qlab, pred)])
    soft = Q[np.arange(len(P)), :, pred].mean(1)
    agree = np.array([r["agreement"] for r in rows])
    return {
        "max_softmax": msp,
        "quadrant_agree_plus_conf": agree + msp,
        "healthy_neutral_plus_conf": (4 - contra) + msp,
        "quadrant_mean_prob": soft,
        "quadrant_mean_prob_plus_conf": 0.5 * soft + 0.5 * msp,
    }, contra


def score_table(rows, classes, reps=1000):
    rows = [r for r in rows if r["true"] in classes]
    y = np.array([classes.index(r["true"]) for r in rows])
    P = np.array([r["probs"] for r in rows])
    correct = P.argmax(1) == y
    sc = dict(ex.scores_for(rows, classes))
    qs, contra = quadrant_scores(rows, classes)
    sc.update(qs)
    msp = sc["max_softmax"]
    out = {"n": len(rows), "errors": int((~correct).sum())}
    if correct.all() or (~correct).all():
        out["note"] = "all correct or all wrong; AURC undefined"
        return out
    for name, s in sc.items():
        out[name] = {"aurc": round(ex.aurc(s, correct), 4),
                     "ci95": ex.bootstrap(lambda i, s=s: ex.aurc(s[i], correct[i]), len(rows), reps)}
        if name != "max_softmax":
            out[name]["minus_msp"] = round(out[name]["aurc"] - ex.aurc(msp, correct), 4)
            out[name]["minus_msp_ci95"] = ex.bootstrap(
                lambda i, s=s: ex.aurc(s[i], correct[i]) - ex.aurc(msp[i], correct[i]), len(rows), reps)
    return out


def mechanism(rows, classes):
    """Why quadrant agreement does or does not separate right from wrong answers."""
    rows = [r for r in rows if r["true"] in classes]
    y = np.array([classes.index(r["true"]) for r in rows])
    P = np.array([r["probs"] for r in rows])
    pred = P.argmax(1)
    correct = pred == y
    agree = np.array([r["agreement"] for r in rows])
    _, contra = quadrant_scores(rows, classes)
    healthy_q = np.array([sum(ac.label_kind(l) == "healthy" for l in r["quad_labels"]) for r in rows])
    hist = lambda v, m: [int(((v == k) & m).sum()) for k in range(5)]
    out = {
        "agreement_hist_correct": hist(agree, correct), "agreement_hist_wrong": hist(agree, ~correct),
        "contradictions_hist_correct": hist(contra, correct), "contradictions_hist_wrong": hist(contra, ~correct),
        "healthy_quadrants_hist_correct": hist(healthy_q, correct), "healthy_quadrants_hist_wrong": hist(healthy_q, ~correct),
        "correct_with_agreement_below_2": int(((agree < ac.MIN_QUADRANT_AGREEMENT) & correct).sum()),
        "wrong_with_agreement_4": int(((agree == 4) & ~correct).sum()),
        "top_wrong_pairs_agreement_4": Counter(f"{classes[a]} -> {classes[b]}" for a, b, m in zip(y, pred, (agree == 4) & ~correct) if m).most_common(5),
        "by_true_class": {},
    }
    for c in sorted(set(y.tolist())):
        m = y == c
        out["by_true_class"][classes[c]] = {
            "n": int(m.sum()), "accuracy": round(float(correct[m].mean()), 4),
            "mean_agreement_correct": round(float(agree[m & correct].mean()), 3) if (m & correct).any() else None,
            "mean_agreement_wrong": round(float(agree[m & ~correct].mean()), 3) if (m & ~correct).any() else None,
            "mean_healthy_quadrants": round(float(healthy_q[m].mean()), 3)}
    return out


def gate_extra(rows, classes, reps=1000):
    """Learned gate (logistic regression, 5-fold CV) with and without the two new quadrant features."""
    rows = [r for r in rows if r["true"] in classes]
    X, c, names = ex.gate_features(rows, classes)
    base = ex.compare_gates(X, c, names, reps)
    qs, contra = quadrant_scores(rows, classes)
    X2 = np.column_stack([X, contra, qs["quadrant_mean_prob"]])
    plus = ex.compare_gates(X2, c, names + ["quadrant_contradictions", "quadrant_mean_prob"], reps)
    return {"learned_gate": base, "learned_gate_plus_new_quadrant_features": plus}


# ------------------------------------------------- before/after comparison
def compare_models(rows_a, rows_b, classes):
    """Paired comparison on the same photographs (matched by file path)."""
    a = {r["file"]: r for r in rows_a if r["true"] in classes}
    b = {r["file"]: r for r in rows_b if r["true"] in classes}
    files = sorted(set(a) & set(b))
    ok_a = [a[f]["pred"] == a[f]["true"] for f in files]
    ok_b = [b[f]["pred"] == b[f]["true"] for f in files]
    out = {"n_paired": len(files), "A": ex.pct(sum(ok_a), len(files)), "B": ex.pct(sum(ok_b), len(files)),
           "mcnemar_A_vs_B": ex.mcnemar_exact(ok_a, ok_b), "per_class_recall": {}}
    for lab in sorted({a[f]["true"] for f in files}):
        fs = [f for f in files if a[f]["true"] == lab]
        out["per_class_recall"][lab] = {"A": ex.pct(sum(a[f]["pred"] == lab for f in fs), len(fs)),
                                        "B": ex.pct(sum(b[f]["pred"] == lab for f in fs), len(fs))}
    out["top_errors_A"] = Counter(f"{a[f]['true']} -> {a[f]['pred']}" for f in files if a[f]["pred"] != a[f]["true"]).most_common(5)
    out["top_errors_B"] = Counter(f"{b[f]['true']} -> {b[f]['pred']}" for f in files if b[f]["pred"] != b[f]["true"]).most_common(5)
    return out


# ------------------------------------------------------------- few-shot
def few_shot_fixed_test(lab_X, lab_y, pool_X, pool_y, test_X, test_y, shots=(0, 5, 10, 20, 50, 100),
                        repeats=5, seed=0):
    """Linear head on frozen ImageNet embeddings: laboratory images plus k field photos per class
    from the POOL; always tested on the same held-out field TEST photos."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    rng = np.random.default_rng(seed)
    classes = sorted(set(test_y.tolist()))

    def fit_eval(Xtr, ytr):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0)).fit(Xtr, ytr)
        return float((m.predict(test_X) == test_y).mean())

    out = {}
    for k in list(shots) + ["all_pool"]:
        accs = []
        for _ in range(1 if k in (0, "all_pool") else repeats):
            if k == "all_pool":
                pick = np.arange(len(pool_y))
            else:
                pick = []
                for c in classes:
                    idx = np.where(pool_y == c)[0]
                    if k and len(idx) >= k:
                        pick += rng.choice(idx, k, replace=False).tolist()
                pick = np.array(pick, int)
            Xtr = np.vstack([lab_X, pool_X[pick]]) if len(pick) else lab_X
            ytr = np.concatenate([lab_y, pool_y[pick]]) if len(pick) else lab_y
            accs.append(fit_eval(Xtr, ytr))
        out[str(k)] = {"accuracy_mean": round(float(np.mean(accs)), 4), "accuracy_sd": round(float(np.std(accs)), 4),
                       "repeats": len(accs)}
    out["field_pool_only"] = {"accuracy_mean": round(fit_eval(pool_X, pool_y), 4), "repeats": 1}
    return out


# --------------------------------------------------------------- Gemini
def probe_gemini(api_key, preferred, test_image_bytes, log=print):
    """Try models in order with a real image assessment call; return the first that answers."""
    names = []
    try:
        from google import genai
        for m in genai.Client(api_key=api_key).models.list():
            n = m.name.split("/")[-1]
            actions = getattr(m, "supported_actions", None) or []
            if "flash" in n and "generateContent" in actions and not re.search(r"tts|image|audio|live|embed", n):
                names.append(n)
    except Exception as e:
        log("could not list models:", str(e)[:150])

    def version_key(n):
        nums = re.findall(r"\d+(?:\.\d+)?", n)
        return ("preview" not in n and "exp" not in n, float(nums[0]) if nums else 0.0, "lite" not in n)

    candidates = []
    for n in list(preferred) + sorted(set(names), key=version_key, reverse=True) + [ac.DEFAULT_GEMINI_MODEL]:
        if n not in candidates:
            candidates.append(n)
    tried = []
    for n in candidates:
        out = ac.assess_leaf_image(test_image_bytes, api_key, n)
        tried.append({"model": n, "ok": bool(out.get("ok")), "error": None if out.get("ok") else str(out)[:200]})
        if out.get("ok"):
            return n, tried
        log(f"  {n}: {str(out)[:120]}")
    return None, tried


def gemini_run(sample, api_key, model, out_path, sleep_s=7.0, max_consecutive_failures=6, log=print):
    """sample: list of {key, file}. Resumable; failed calls and calls made with another model are retried."""
    done = {}
    if Path(out_path).exists():
        done = {d["key"]: d for d in json.load(open(out_path)) if d.get("ok") and d.get("model") == model}
    fails = 0
    for i, s in enumerate(sample):
        if s["key"] in done:
            continue
        with Image.open(s["file"]) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")
        out = ac.assess_leaf_image(ac.image_for_assessment(img), api_key, model)
        if not out.get("ok") and "429" in str(out):
            time.sleep(60)
            out = ac.assess_leaf_image(ac.image_for_assessment(img), api_key, model)
        rec = {"key": s["key"], "file": s["file"], "model": model, "ok": bool(out.get("ok"))}
        if out.get("ok"):
            rec["assessment"] = out["assessment"]
            fails = 0
        else:
            rec["error"] = str(out)[:300]
            fails += 1
        done[s["key"]] = rec
        json.dump(list(done.values()), open(out_path, "w"), indent=1, default=str)
        if i % 10 == 0:
            log(f"gemini {i}/{len(sample)}")
        if fails >= max_consecutive_failures:
            log(f"stopping: {fails} failures in a row; last error: {rec['error'][:200]}")
            break
        time.sleep(sleep_s)
    return list(done.values())


def gemini_compare(records, rows_by_file):
    """Turn raw assessments into exp_lib's record format for one model's predictions, then analyse."""
    recs, rows_by_key = [], {}
    for g in records:
        row = rows_by_file.get(g["file"])
        if row is None:
            continue
        rows_by_key[g["key"]] = row
        rec = {"key": g["key"], "ok": g.get("ok", False)}
        if g.get("ok"):
            a = g["assessment"]
            cmp_ = ac.compare_assessments(row["model_crop"], {"primary": row["pred"], "verdict_kind": row["verdict_kind"]}, a)
            rec.update({"g_crop": a["crop"], "g_label": a["closest_label"], "g_certainty": a["certainty"],
                        "category": cmp_["category"], "supports_prescription": cmp_["supports_prescription"]})
        recs.append(rec)
    return ex.gemini_analyse(recs, rows_by_key) if recs else {"assessed": 0}

Writing /content/followup_lib.py


In [7]:
# 2. Data: laboratory rice data (same as the first run), router, deployed model, Sethy field photos
import importlib, kagglehub, gdown
import exp_lib as ex, agri_core as ac, train_rice as tr, followup_lib as fl
from agri_core import RICE_CLASSES
from PIL import Image, ImageOps
rice_root = Path(kagglehub.dataset_download('anshulm257/rice-disease-dataset'))
weed_root = Path(kagglehub.dataset_download('fpeccia/weed-detection-in-soybean-crops'))
rice_data = rice_root
while len([p for p in rice_data.iterdir() if p.is_dir()]) == 1:
    rice_data = [p for p in rice_data.iterdir() if p.is_dir()][0]
DATA = Path('/content/rice_data')
if not DATA.exists():
    DATA.mkdir()
    targets = {ex.norm(c.replace('Rice_', '')): c for c in RICE_CLASSES if c.startswith('Rice_')}
    targets['healthy_rice_leaf'] = 'Rice_Healthy'
    for folder in sorted(p for p in rice_data.iterdir() if p.is_dir()):
        key = ex.norm(folder.name)
        if key in targets: shutil.copytree(folder, DATA / targets[key])
        else: print('SKIPPED rice folder:', folder.name)
    soy = weed_root / 'dataset'
    for src, dst in [('broadleaf', 'Weed_Broadleaf'), ('grass', 'Weed_Grass')]:
        (DATA / dst).mkdir()
        for f in (soy / src).rglob('*'):
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'): shutil.copy(f, DATA / dst / f.name)
            elif f.suffix.lower() in ('.tif', '.tiff', '.bmp', '.webp'): Image.open(f).convert('RGB').save(DATA / dst / (f.stem + '.png'))
print({c: len(ex.list_images(DATA / c)) for c in RICE_CLASSES})
for fid, name in [('1p2vZgq_FBigVnlhQPLQD4w2yjDn4zus3', 'old_rice.tflite'), ('10LuRi-n6wFu4um9deAWBu0sr0-8e1MLV', 'router.tflite')]:
    if not Path(f'/content/{name}').exists(): gdown.download(id=fid, output=f'/content/{name}', quiet=True)
# the lab splits saved by the first run must point at files that exist again now
for s in SEEDS:
    sp = json.load(open(PREV / f'rice_seed{s}' / 'split.json'))
    allp = [p for part in sp.values() for p, _ in part]
    miss = sum(not Path(p).exists() for p in allp)
    print(f'seed {s}: lab split has {len(allp)} images, {miss} missing')
    if miss > 0.01 * len(allp):
        raise RuntimeError(f'STOP: {miss} images of the seed-{s} lab split are missing; the laboratory download changed.')
FIELD = Path('/content/sethy_field')
src = kagglehub.dataset_download('nirmalsankalana/rice-leaf-disease-image')
counts, unmatched = fl.stage_sethy(src, FIELD)
print('Sethy field photos staged:', counts, '| folders left out:', unmatched)
assert sum(counts.values()) > 1000, 'too few field photos staged - check the folder names above'
# optional extra test sets (different source from everything used for training)
EXTRA = {}
dhan, msg = fl.find_dhan(BASE / 'agri_external', Path('/content/dhan'))
print('Dhan-Shomadhan (Bangladesh):', msg)
if dhan:
    kept, removed = ex.dedupe_against(dhan, ex.list_images(DATA), DUP_THRESHOLD)
    EXTRA['dhan_field'] = {'crop': 'Rice', 'items': sorted(kept, key=lambda x: str(x[0]))[:OTHER_SET_CAP], 'removed_lab_copies': len(removed)}
for crop, classes in (('rice', ac.RICE_CLASSES), ('maize', ac.CORN_CLASSES)):
    own, unknown = fl.own_photos(BASE / 'agri_own_photos' / crop, classes)
    if unknown: print(f'own {crop} photos: folders not recognised (rename to one of {classes}):', unknown)
    if own:
        EXTRA[f'own_{crop}'] = {'crop': crop.capitalize(), 'items': sorted(own, key=lambda x: str(x[0]))[:OTHER_SET_CAP]}
print('own-photo folders expected: rice ->', ac.RICE_CLASSES, '| maize ->', ac.CORN_CLASSES)
print('extra test sets:', {k: len(v['items']) for k, v in EXTRA.items()} or 'none')
if EXTRA.get('own_maize') and not Path('/content/maize.tflite').exists():
    gdown.download(id='1_1PcQqUFFiK9tgpXwivM6J7OJShL18jk', output='/content/maize.tflite', quiet=True)
save('extra_sets_report.json', {k: {'n': len(v['items']), 'classes': dict(__import__('collections').Counter(l for _, l in v['items'])),
                                    **({'removed_lab_copies': v['removed_lab_copies']} if 'removed_lab_copies' in v else {})} for k, v in EXTRA.items()})

Using Colab cache for faster access to the 'rice-disease-dataset' dataset.


100%|██████████| 2.37G/2.37G [02:28<00:00, 17.0MB/s]

Extracting files...


{'Rice_Bacterial_Leaf_Blight': 636, 'Rice_Brown_Spot': 646, 'Rice_Healthy': 653, 'Rice_Leaf_Blast': 634, 'Rice_Leaf_scald': 628, 'Rice_Sheath_Blight': 632, 'Weed_Broadleaf': 1191, 'Weed_Grass': 3520}
seed 42: lab split has 8540 images, 0 missing
seed 7: lab split has 8540 images, 0 missing
seed 2024: lab split has 8540 images, 0 missing


100%|██████████| 195M/195M [00:09<00:00, 20.7MB/s]

Extracting files...


Sethy field photos staged: {'Rice_Bacterial_Leaf_Blight': 1584, 'Rice_Leaf_Blast': 1440, 'Rice_Brown_Spot': 1600} | folders left out: {}
Dhan-Shomadhan (Bangladesh): not found: download https://data.mendeley.com/datasets/znsxdctwtt/1 in a browser and put the zip in My Drive/agri_external
own-photo folders expected: rice -> ['Rice_Bacterial_Leaf_Blight', 'Rice_Brown_Spot', 'Rice_Healthy', 'Rice_Leaf_Blast', 'Rice_Leaf_scald', 'Rice_Sheath_Blight', 'Weed_Broadleaf', 'Weed_Grass'] | maize -> ['Maize_Blight', 'Maize_Common_Rust', 'Maize_Gray_Leaf_Spot', 'Maize_Healthy', 'Weed_Broadleaf', 'Weed_Grass']
extra test sets: none


In [8]:
# 3. Field test / field pool split (computed once, saved, reused on every re-run)
if done('field_split.json'):
    FS = json.load(open(OUT / 'field_split.json'))
    assert all(Path(p).exists() for p, _ in FS['test'][:50]), 'saved split points at missing files'
else:
    items = [(p, c) for c in fl.FIELD_CLASSES for p in ex.list_images(FIELD / c)]
    FS = fl.field_split(items, ex.list_images(DATA), DUP_THRESHOLD, FIELD_TEST_FRACTION, FIELD_TEST_PER_CLASS, SPLIT_SEED)
    save('field_split.json', FS)
print(json.dumps(FS['report'], indent=1))
field_test = [(Path(p), l) for p, l in FS['test']]

{
 "field_photos_found": 4624,
 "removed_as_copies_of_lab_images": 349,
 "after_dedupe": 4275,
 "near_duplicate_groups_in_field_set": 1264,
 "test_fraction": 0.3,
 "test_cap_per_class": 300,
 "test_used": 900,
 "test_left_unused": 392,
 "pool": 2983,
 "test_class_counts": {
  "Rice_Bacterial_Leaf_Blight": 300,
  "Rice_Brown_Spot": 300,
  "Rice_Leaf_Blast": 300
 },
 "pool_class_counts": {
  "Rice_Bacterial_Leaf_Blight": 1081,
  "Rice_Brown_Spot": 937,
  "Rice_Leaf_Blast": 965
 },
 "seed": 0
}


In [9]:
# 4. Field adaptation: retrain each seed on its own lab training images + the field pool (~25 min per seed)
from types import SimpleNamespace
for seed in SEEDS:
    run = OUT / f'adapted_seed{seed}'
    if (run / 'metrics.json').exists():
        print('seed', seed, 'already done'); continue
    run.mkdir(exist_ok=True); t0 = time.time()
    lab = json.load(open(PREV / f'rice_seed{seed}' / 'split.json'))
    tr_idx, va_idx = fl.pool_train_val(FS['pool'], FS['pool_groups'], FIELD_VAL_FRACTION, seed)
    items, splits = [], {'train': [], 'val': [], 'test': []}
    for part in ('train', 'val', 'test'):
        for p, l in lab[part]:
            splits[part].append(len(items)); items.append((Path(p), l))
    for part, idx in (('train', tr_idx), ('val', va_idx)):
        for i in idx:
            p, l = FS['pool'][i]; splits[part].append(len(items)); items.append((Path(p), l))
    t1 = time.time()
    small = fl.shrink_for_training([p for p, _ in items], '/content/train_cache')
    items = [(Path(small[str(p)]), l) for p, l in items]
    print(f'seed {seed}: {len(items)} images ready for training ({sum(str(p).startswith("/content/train_cache") for p, _ in items)} shrunk) in {time.time()-t1:.0f}s')
    args = SimpleNamespace(seed=seed, batch=32, weights='imagenet', dropout=0.3, patience=5,
                           epochs_head=15, epochs_finetune=20, unfreeze=40)
    tf.keras.backend.clear_session()
    model, history = tr.train(tf, args, items, splits)
    path, inference = tr.export_tflite(tf, model, run)
    export = tr.verify_export(path, inference, items, splits['test'])
    predict, _ = tr.tflite_predictor(path)
    lab_test = tr.evaluate(predict, items, splits['test'])
    json.dump({'seed': seed, 'history': history, 'export': export, 'lab_test': lab_test,
               'train_images': {'lab': len(lab['train']), 'field_pool': len(tr_idx)},
               'val_images': {'lab': len(lab['val']), 'field_pool': len(va_idx)},
               'minutes': round((time.time() - t0) / 60, 1)}, open(run / 'metrics.json', 'w'), indent=1, default=float)
    print(f'seed {seed}: lab test accuracy {lab_test["accuracy"]:.4f} | {(time.time()-t0)/60:.1f} min')
    del model, inference

seed 42: 11523 images ready for training (6059 shrunk) in 7s
24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/15
225/225 - 26s - 114ms/step - accuracy: 0.6563 - loss: 1.0074 - val_accuracy: 0.8608 - val_loss: 0.4164 - learning_rate: 0.0010
Epoch 2/15
225/225 - 6s - 26ms/step - accuracy: 0.7871 - loss: 0.6314 - val_accuracy: 0.8818 - val_loss: 0.3475 - learning_rate: 0.0010
Epoch 3/15
225/225 - 6s - 26ms/step - accuracy: 0.8250 - loss: 0.5309 - val_accuracy: 0.8934 - val_loss: 0.3060 - learning_rate: 0.0010
Epoch 4/15
225/225 - 6s - 27ms/step - accuracy: 0.8315 - loss: 0.4890 - val_accuracy: 0.9025 - val_loss: 0.2860 - learning_rate: 0.0010
Epoch 5/15
225/225 - 6s - 26ms/step - accuracy: 0.8442 - loss: 0.4665 - val_accuracy: 0.9036 - val_loss: 0.2765 - learning_rate: 0.0010
Epoch 6/15
225/225 - 6s - 26ms/step - accuracy: 0.8469 - loss: 0.4467 - val_accuracy: 0.9161 - val_loss: 0.2605 - learning_rate: 0.0010
Epoch 7/15
225/225 - 6s - 26ms/step - accuracy: 0.8533 - loss: 0.4215 

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


seed 42: lab test accuracy 0.9245 | 5.6 min
seed 7: 11523 images ready for training (6059 shrunk) in 1s
Epoch 1/15
219/219 - 19s - 89ms/step - accuracy: 0.6617 - loss: 1.0104 - val_accuracy: 0.8420 - val_loss: 0.4494 - learning_rate: 0.0010
Epoch 2/15
219/219 - 6s - 26ms/step - accuracy: 0.7975 - loss: 0.6231 - val_accuracy: 0.8911 - val_loss: 0.3323 - learning_rate: 0.0010
Epoch 3/15
219/219 - 6s - 26ms/step - accuracy: 0.8217 - loss: 0.5311 - val_accuracy: 0.9000 - val_loss: 0.3035 - learning_rate: 0.0010
Epoch 4/15
219/219 - 6s - 26ms/step - accuracy: 0.8392 - loss: 0.4907 - val_accuracy: 0.9031 - val_loss: 0.2844 - learning_rate: 0.0010
Epoch 5/15
219/219 - 6s - 26ms/step - accuracy: 0.8480 - loss: 0.4591 - val_accuracy: 0.9166 - val_loss: 0.2600 - learning_rate: 0.0010
Epoch 6/15
219/219 - 6s - 26ms/step - accuracy: 0.8498 - loss: 0.4392 - val_accuracy: 0.9058 - val_loss: 0.2627 - learning_rate: 0.0010
Epoch 7/15
219/219 - 6s - 26ms/step - accuracy: 0.8591 - loss: 0.4158 - val_acc

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


seed 7: lab test accuracy 0.8856 | 5.1 min
seed 2024: 11523 images ready for training (6059 shrunk) in 1s
Epoch 1/15
211/211 - 19s - 89ms/step - accuracy: 0.6600 - loss: 1.0189 - val_accuracy: 0.7385 - val_loss: 0.7436 - learning_rate: 0.0010
Epoch 2/15
211/211 - 5s - 24ms/step - accuracy: 0.7861 - loss: 0.6371 - val_accuracy: 0.7777 - val_loss: 0.6065 - learning_rate: 0.0010
Epoch 3/15
211/211 - 5s - 24ms/step - accuracy: 0.8141 - loss: 0.5468 - val_accuracy: 0.7991 - val_loss: 0.5418 - learning_rate: 0.0010
Epoch 4/15
211/211 - 5s - 24ms/step - accuracy: 0.8342 - loss: 0.4898 - val_accuracy: 0.8096 - val_loss: 0.5044 - learning_rate: 0.0010
Epoch 5/15
211/211 - 5s - 24ms/step - accuracy: 0.8477 - loss: 0.4577 - val_accuracy: 0.8240 - val_loss: 0.4574 - learning_rate: 0.0010
Epoch 6/15
211/211 - 5s - 24ms/step - accuracy: 0.8509 - loss: 0.4376 - val_accuracy: 0.8316 - val_loss: 0.4472 - learning_rate: 0.0010
Epoch 7/15
211/211 - 5s - 24ms/step - accuracy: 0.8583 - loss: 0.4181 - val_a

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


seed 2024: lab test accuracy 0.9415 | 5.5 min


In [10]:
# 5. Inference on the field test photos with the app's own decision code (+ per-quadrant records)
def interp(path, classes, name):
    it = tf.lite.Interpreter(model_path=str(path)); it.allocate_tensors()
    ac.check_output_classes(it, classes, name); return it
router = interp('/content/router.tflite', ac.ROUTER_CLASSES, 'Router')
MODELS = {}
if EVAL_DEPLOYED_MODEL: MODELS['deployed'] = '/content/old_rice.tflite'
for s in SEEDS:
    MODELS[f'lab_seed{s}'] = PREV / f'rice_seed{s}' / 'rice_model.tflite'
    MODELS[f'adapted_seed{s}'] = OUT / f'adapted_seed{s}' / 'rice_model.tflite'
for key, path in MODELS.items():
    fname = f'rows_field_test__{key}.json'
    if done(fname): print('done', fname); continue
    rows = fl.eval_model(field_test, interp(path, RICE_CLASSES, 'Rice'), RICE_CLASSES, router, key)
    save(fname, rows)
# extra sets: every rice model on Bangladesh / own rice photos; the maize model on own maize photos
for set_name, spec in EXTRA.items():
    keys = ['maize'] if spec['crop'] == 'Maize' else list(MODELS)
    for key in keys:
        fname = f'rows_{set_name}__{key}.json'
        if done(fname): print('done', fname); continue
        if key == 'maize':
            it_, classes = interp('/content/maize.tflite', ac.CORN_CLASSES, 'Maize'), ac.CORN_CLASSES
        else:
            it_, classes = interp(MODELS[key], RICE_CLASSES, 'Rice'), RICE_CLASSES
        save(fname, fl.eval_model(spec['items'], it_, classes, router, f'{set_name}__{key}', crop=spec['crop']))
# laboratory test split of each seed: lab-only vs adapted (forgetting check)
if not done('lab_test_check.json'):
    chk = {}
    for s in SEEDS:
        lab = json.load(open(PREV / f'rice_seed{s}' / 'split.json'))
        items = [(Path(p), l) for p, l in lab['test']]; idx = list(range(len(items)))
        chk[s] = {}
        for k in ('lab', 'adapted'):
            path = MODELS[f'{k}_seed{s}']; predict, _ = tr.tflite_predictor(path)
            m = tr.evaluate(predict, items, idx)
            chk[s][k] = {'accuracy': m['accuracy'], 'macro_f1': m['per_class']['macro avg']['f1-score'],
                         'per_class_recall': {c: m['per_class'][c]['recall'] for c in RICE_CLASSES}, 'n': m['test_images']}
    save('lab_test_check.json', chk)
print(json.dumps(json.load(open(OUT / 'lab_test_check.json')), indent=1)[:1500])

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


deployed 0 / 900 0s
deployed 150 / 900 36s
deployed 300 / 900 72s
deployed 450 / 900 107s
deployed 600 / 900 142s
deployed 750 / 900 178s
lab_seed42 0 / 900 0s
lab_seed42 150 / 900 47s
lab_seed42 300 / 900 94s
lab_seed42 450 / 900 141s
lab_seed42 600 / 900 189s
lab_seed42 750 / 900 236s
adapted_seed42 0 / 900 0s
adapted_seed42 150 / 900 48s
adapted_seed42 300 / 900 95s
adapted_seed42 450 / 900 142s
adapted_seed42 600 / 900 188s
adapted_seed42 750 / 900 234s
lab_seed7 0 / 900 0s
lab_seed7 150 / 900 47s
lab_seed7 300 / 900 94s
lab_seed7 450 / 900 142s
lab_seed7 600 / 900 189s
lab_seed7 750 / 900 237s
adapted_seed7 0 / 900 0s
adapted_seed7 150 / 900 47s
adapted_seed7 300 / 900 94s
adapted_seed7 450 / 900 142s
adapted_seed7 600 / 900 188s
adapted_seed7 750 / 900 236s
lab_seed2024 0 / 900 0s
lab_seed2024 150 / 900 48s
lab_seed2024 300 / 900 95s
lab_seed2024 450 / 900 142s
lab_seed2024 600 / 900 190s
lab_seed2024 750 / 900 237s
adapted_seed2024 0 / 900 0s
adapted_seed2024 150 / 900 47s
adapt

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use 

{
 "42": {
  "lab": {
   "accuracy": 0.9331416247304097,
   "macro_f1": 0.894101244576443,
   "per_class_recall": {
    "Rice_Bacterial_Leaf_Blight": 0.8699186991869918,
    "Rice_Brown_Spot": 0.8790322580645161,
    "Rice_Healthy": 0.9923664122137404,
    "Rice_Leaf_Blast": 0.8990825688073395,
    "Rice_Leaf_scald": 0.6336633663366337,
    "Rice_Sheath_Blight": 0.8854166666666666,
    "Weed_Broadleaf": 1.0,
    "Weed_Grass": 0.9964349376114082
   },
   "n": 1391
  },
  "adapted": {
   "accuracy": 0.9237958303378864,
   "macro_f1": 0.8815515441282219,
   "per_class_recall": {
    "Rice_Bacterial_Leaf_Blight": 0.6422764227642277,
    "Rice_Brown_Spot": 0.8225806451612904,
    "Rice_Healthy": 0.9923664122137404,
    "Rice_Leaf_Blast": 0.8623853211009175,
    "Rice_Leaf_scald": 0.8613861386138614,
    "Rice_Sheath_Blight": 0.9166666666666666,
    "Weed_Broadleaf": 1.0,
    "Weed_Grass": 0.9964349376114082
   },
   "n": 1391
  }
 },
 "7": {
  "lab": {
   "accuracy": 0.8918287937743191,
   

In [11]:
# 6. Analysis: before/after, gates, quadrant mechanism
rows = {k: json.load(open(OUT / f'rows_field_test__{k}.json')) for k in MODELS}
summary = {'field_test_n': len(field_test), 'per_seed': {}}
for s in SEEDS:
    summary['per_seed'][s] = fl.compare_models(rows[f'lab_seed{s}'], rows[f'adapted_seed{s}'], RICE_CLASSES)
accs = lambda k: [summary['per_seed'][s][k]['pct'] for s in SEEDS]
summary['lab_only_mean_sd'] = [round(float(np.mean(accs('A'))), 2), round(float(np.std(accs('A'), ddof=1)), 2)]
summary['adapted_mean_sd'] = [round(float(np.mean(accs('B'))), 2), round(float(np.std(accs('B'), ddof=1)), 2)]
if 'deployed' in rows:
    summary['deployed_vs_adapted_seed42'] = fl.compare_models(rows['deployed'], rows['adapted_seed42'], RICE_CLASSES)
save('adaptation_summary.json', summary)
analysis, gates, mech = {}, {}, {}
for k, r in rows.items():
    analysis[k] = ex.analyse(r, RICE_CLASSES, reps=1000)
    analysis[k]['scores_incl_quadrant_variants'] = fl.score_table(r, RICE_CLASSES, reps=1000)
    mech[k] = fl.mechanism(r, RICE_CLASSES)
    rr = [x for x in r if x['true'] in RICE_CLASSES]
    y = np.array([RICE_CLASSES.index(x['true']) for x in rr]); P = np.array([x['probs'] for x in rr])
    gates[k] = fl.gate_extra(r, RICE_CLASSES)
    gates[k]['conformal_sets_90'] = ex.conformal_sets(P, y, 0.10)
    gates[k]['conformal_sets_95'] = ex.conformal_sets(P, y, 0.05)
extra = {}
for set_name in EXTRA:
    extra[set_name] = {}
    for f in sorted(OUT.glob(f'rows_{set_name}__*.json')):
        key = f.stem.split('__')[1]; r = json.load(open(f)); classes = ac.CORN_CLASSES if key == 'maize' else RICE_CLASSES
        extra[set_name][key] = {'analysis': ex.analyse(r, classes, reps=1000), 'mechanism': fl.mechanism(r, classes) if r else None}
    if set_name != 'own_maize':
        extra[set_name]['lab_vs_adapted'] = {s: fl.compare_models(json.load(open(OUT / f'rows_{set_name}__lab_seed{s}.json')),
                                                                  json.load(open(OUT / f'rows_{set_name}__adapted_seed{s}.json')), RICE_CLASSES) for s in SEEDS}
save('extra_sets_analysis.json', extra)
save('analysis_followup.json', analysis); save('gates_followup.json', gates); save('quadrant_mechanism.json', mech)

print('FIELD TEST, n =', len(field_test))
for s in SEEDS:
    c = summary['per_seed'][s]
    print(f"seed {s}: lab-only {c['A']['pct']}% {c['A']['ci95']}  ->  adapted {c['B']['pct']}% {c['B']['ci95']}   McNemar p={c['mcnemar_A_vs_B']}")
print('mean ± sd: lab-only', summary['lab_only_mean_sd'], '| adapted', summary['adapted_mean_sd'])
print('\nAURC (lower is better): max-softmax | quadrant agreement | healthy-neutral | learned gate')
for k in rows:
    t = analysis[k]['scores_incl_quadrant_variants']
    if 'max_softmax' not in t: print(k, t.get('note')); continue
    lg = gates[k]['learned_gate']
    print(f"{k:18s} acc {analysis[k]['accuracy']['pct']:5}%  {t['max_softmax']['aurc']} | {t['quadrant_agree_plus_conf']['aurc']} "
          f"| {t['healthy_neutral_plus_conf']['aurc']} | {lg.get('aurc_learned_gate')}")
for set_name, v in extra.items():
    print(f'\n== {set_name}:', {k: a['analysis'].get('accuracy', {}).get('pct') for k, a in v.items() if k != 'lab_vs_adapted'})

FIELD TEST, n = 900
seed 42: lab-only 27.0% [24.2, 30.0]  ->  adapted 97.33% [96.1, 98.2]   McNemar p={'a_only_correct': 1, 'b_only_correct': 634, 'p_exact': 8.921540774203804e-189}
seed 7: lab-only 22.0% [19.4, 24.8]  ->  adapted 97.56% [96.3, 98.4]   McNemar p={'a_only_correct': 1, 'b_only_correct': 681, 'p_exact': 6.807594151599639e-203}
seed 2024: lab-only 20.56% [18.0, 23.3]  ->  adapted 96.56% [95.2, 97.6]   McNemar p={'a_only_correct': 2, 'b_only_correct': 686, 'p_exact': 3.6912416339442065e-202}
mean ± sd: lab-only [23.19, 3.38] | adapted [97.15, 0.52]

AURC (lower is better): max-softmax | quadrant agreement | healthy-neutral | learned gate
deployed           acc  27.0%  0.5663 | 0.7476 | 0.4965 | 0.5118
lab_seed42         acc  27.0%  0.592 | 0.8437 | 0.5415 | 0.4359
adapted_seed42     acc 97.33%  0.0043 | 0.0046 | 0.0088 | 0.0029
lab_seed7          acc  22.0%  0.7281 | 0.8827 | 0.6632 | 0.5096
adapted_seed7      acc 97.56%  0.0006 | 0.0036 | 0.0036 | 0.0006
lab_seed2024      

In [12]:
# 7. Rice few-shot curve (frozen ImageNet EfficientNetV2-B0 embeddings + linear head)
if done('few_shot_rice.json'):
    print(json.dumps(json.load(open(OUT / 'few_shot_rice.json')), indent=1))
else:
    from PIL import ImageOps
    backbone = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet', input_shape=(224, 224, 3),
                                                       include_preprocessing=True, pooling='avg')
    def embed(pairs, batch=64):
        X, keep = [], []
        for i in range(0, len(pairs), batch):
            arrs = []
            for p, l in pairs[i:i + batch]:
                try:
                    with Image.open(p) as im:
                        arrs.append(((p, l), np.asarray(ImageOps.exif_transpose(im).convert('RGB').resize((224, 224)), dtype=np.float32)))
                except Exception:
                    pass
            if arrs:
                X.append(backbone.predict(np.stack([a for _, a in arrs]), verbose=0)); keep += [pl for pl, _ in arrs]
        return np.vstack(X), np.array([l for _, l in keep])
    lab_pairs = []
    for c in fl.FIELD_CLASSES:
        ps = sorted(ex.list_images(DATA / c)); random.Random(0).shuffle(ps); lab_pairs += [(p, c) for p in ps[:300]]
    LX, Ly = embed(lab_pairs); PX, Py = embed([(Path(p), l) for p, l in FS['pool']]); TX, Ty = embed(field_test)
    few = {'lab_images': len(Ly), 'pool_images': len(Py), 'test_images': len(Ty), 'classes': fl.FIELD_CLASSES,
           'note': '3-class linear head; not the same as the 8-class deployed model',
           'curve': fl.few_shot_fixed_test(LX, Ly, PX, Py, TX, Ty, FEW_SHOTS)}
    save('few_shot_rice.json', few); print(json.dumps(few, indent=1))

{
 "lab_images": 900,
 "pool_images": 2983,
 "test_images": 900,
 "classes": [
  "Rice_Bacterial_Leaf_Blight",
  "Rice_Brown_Spot",
  "Rice_Leaf_Blast"
 ],
 "note": "3-class linear head; not the same as the 8-class deployed model",
 "curve": {
  "0": {
   "accuracy_mean": 0.4544,
   "accuracy_sd": 0.0,
   "repeats": 1
  },
  "5": {
   "accuracy_mean": 0.5656,
   "accuracy_sd": 0.0306,
   "repeats": 5
  },
  "10": {
   "accuracy_mean": 0.6349,
   "accuracy_sd": 0.0198,
   "repeats": 5
  },
  "20": {
   "accuracy_mean": 0.7056,
   "accuracy_sd": 0.0215,
   "repeats": 5
  },
  "50": {
   "accuracy_mean": 0.8109,
   "accuracy_sd": 0.0072,
   "repeats": 5
  },
  "100": {
   "accuracy_mean": 0.8747,
   "accuracy_sd": 0.0219,
   "repeats": 5
  },
  "all_pool": {
   "accuracy_mean": 0.9889,
   "accuracy_sd": 0.0,
   "repeats": 1
  },
  "field_pool_only": {
   "accuracy_mean": 0.9922,
   "repeats": 1
  }
 }
}


In [13]:
# 8. Gemini cross-check (maize field photos + rice field test photos)
try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_KEY = None
if not RUN_GEMINI:
    print('Gemini step switched off in Settings.')
elif not GEMINI_KEY:
    print("No GEMINI_API_KEY for this notebook: key icon > GEMINI_API_KEY > switch on 'Notebook access', then run this cell again.")
else:
    if not Path('/content/PlantDoc').exists():
        !git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset /content/PlantDoc
    maize_rows = json.load(open(PREV / 'rows_maize_field__maize.json'))
    maize_rows = [r for r in maize_rows if Path(r.get('file', '')).exists()]
    print('maize field photos available:', len(maize_rows))
    rng = random.Random(0)
    m_sample = sorted(maize_rows, key=lambda r: r['file']); rng.shuffle(m_sample); m_sample = m_sample[:GEMINI_MAIZE_SAMPLE]
    r_sample = []
    for c in fl.FIELD_CLASSES:
        ps = sorted(str(p) for p, l in field_test if l == c); rng.shuffle(ps); r_sample += ps[:GEMINI_RICE_PER_CLASS]
    sample = [{'key': 'maize:' + r['file'], 'file': r['file']} for r in m_sample] + [{'key': 'rice:' + f, 'file': f} for f in r_sample]
    with Image.open(sample[0]['file']) as im:
        probe_bytes = ac.image_for_assessment(ImageOps.exif_transpose(im).convert('RGB'))
    model, tried = fl.probe_gemini(GEMINI_KEY, GEMINI_PREFERRED, probe_bytes)
    save('gemini_model_used.json', {'model': model, 'tried': tried})
    if not model:
        print('No Gemini model answered for this key - see gemini_model_used.json.')
    else:
        print('Gemini model:', model, f'| {len(sample)} photos, about {len(sample) * 8 // 60} min')
        recs = fl.gemini_run(sample, GEMINI_KEY, model, OUT / 'gemini_records.json')
        by_file = lambda rs: {r['file']: r for r in rs}
        g = {'model': model, 'calls_ok': sum(r['ok'] for r in recs), 'calls_failed': sum(not r['ok'] for r in recs),
             'maize_field__maize': fl.gemini_compare([r for r in recs if r['key'].startswith('maize:')], by_file(maize_rows))}
        for k in ('lab_seed42', 'adapted_seed42', 'deployed'):
            if k in rows:
                g[f'rice_field_test__{k}'] = fl.gemini_compare([r for r in recs if r['key'].startswith('rice:')], by_file(rows[k]))
        save('gemini_analysis.json', g)
        for k, v in g.items():
            if isinstance(v, dict): print(k, json.dumps({x: v.get(x) for x in ('assessed', 'failed_calls', 'ml_accuracy_on_sample', 'gemini_label_accuracy', 'gemini_crop_accuracy', 'fisher_p')}))

maize field photos available: 83
could not list models: Cannot send a request, as the client has been closed.
  gemini-3.8-flash: {'ok': False, 'kind': 'other', 'message': 'The AI service is temporarily unavailable. The image diagnosis above is unaff
  gemini-2.5-flash: {'ok': False, 'kind': 'model', 'message': "The configured Gemini model is not available to this key. Check gemini_model 
No Gemini model answered for this key - see gemini_model_used.json.


In [14]:
# 9. Package everything for download (models are left on Drive, not zipped)
zip_path = BASE / 'agri_results_followup.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file() and f.suffix != '.tflite':
            z.write(f, f.relative_to(OUT.parent))
print('Saved', zip_path, f'({zip_path.stat().st_size / 1e6:.1f} MB). Download it from My Drive and send it back.')

Saved /content/drive/MyDrive/agri_results_followup.zip (1.5 MB). Download it from My Drive and send it back.
